# Hypothesis 1: Forest Sketch improves predictive performance

This notebook compares original features, initial random projection, one-shot tree-path projection, and full Forest Sketch at the same final dimension. It reports classification accuracy and regression R² over held-out test data.

In [1]:
import sys
from pathlib import Path

notebooks_dir = Path.cwd() / 'notebooks'
if not (notebooks_dir / '_hypothesis_utils.py').exists():
    notebooks_dir = Path.cwd()
sys.path.insert(0, str(notebooks_dir))

import pandas as pd
from _hypothesis_utils import (
    CLASSIFICATION_SEEDS, OneShotTreePathTransformer, classification_data,
    classification_score, downstream_classifier, downstream_regressor,
    forest_classifier, forest_regressor, forest_sketch, initial_projection,
    regression_data, regression_scores,
)

In [2]:
classification_rows = []
for seed in CLASSIFICATION_SEEDS[:2]:
    X_train, _, X_test, y_train, _, y_test = classification_data(seed)
    X_train_0, X_test_0, _ = initial_projection(X_train, X_test, 32, seed)
    models = {
        'original': ('array', X_train, X_test),
        'initial random projection': ('array', X_train_0, X_test_0),
        'one-shot tree-path': ('transformer', OneShotTreePathTransformer(forest_classifier(seed), 32, seed), None),
        'Forest Sketch': ('transformer', forest_sketch(seed, 32, 2), None),
    }
    for name, (kind, train_view, test_view) in models.items():
        if kind == 'transformer':
            representation = train_view
            representation.fit(X_train, y_train)
            train_view = representation.transform(X_train)
            test_view = representation.transform(X_test)
        accuracy, errors = classification_score(
            downstream_classifier(seed), train_view, y_train, test_view, y_test
        )
        classification_rows.append({'seed': seed, 'method': name, 'accuracy': accuracy, 'errors': errors})
classification = pd.DataFrame(classification_rows)
display(classification.round(3))
classification_summary = classification.groupby('method').accuracy.agg(['mean', 'std']).round(3)
display(classification_summary)

,seed,method,accuracy,errors
0,0,original,0.620,167
1,0,initial random projection,0.611,171
2,0,one-shot tree-path,0.661,149
3,0,Forest Sketch,0.605,174
4,1,original,0.680,141
5,1,initial random projection,0.684,139
6,1,one-shot tree-path,0.782,96
7,1,Forest Sketch,0.666,147


,mean,std
method,,
Forest Sketch,0.635,0.043
initial random projection,0.648,0.051
one-shot tree-path,0.722,0.085
original,0.650,0.042


In [3]:
regression_rows = []
for seed in CLASSIFICATION_SEEDS[:2]:
    X_train, _, X_test, y_train, _, y_test = regression_data(seed)
    X_train_0, X_test_0, _ = initial_projection(X_train, X_test, 32, seed)
    for name, train_view, test_view in [
        ('original', X_train, X_test),
        ('initial random projection', X_train_0, X_test_0),
    ]:
        r2, rmse = regression_scores(downstream_regressor(), train_view, y_train, test_view, y_test)
        regression_rows.append({'seed': seed, 'method': name, 'r2': r2, 'rmse': rmse})
    for name, transformer in [
        ('one-shot tree-path', OneShotTreePathTransformer(forest_regressor(seed), 32, seed)),
        ('Forest Sketch', forest_sketch(seed, 32, 2, kind='regressor')),
    ]:
        train_view = transformer.fit_transform(X_train, y_train)
        test_view = transformer.transform(X_test)
        r2, rmse = regression_scores(downstream_regressor(), train_view, y_train, test_view, y_test)
        regression_rows.append({'seed': seed, 'method': name, 'r2': r2, 'rmse': rmse})
regression = pd.DataFrame(regression_rows)
display(regression.round(3))
regression_summary = regression.groupby('method')[['r2', 'rmse']].mean().round(3)
display(regression_summary)
classification_supported = classification_summary.loc['Forest Sketch', 'mean'] >= classification_summary.loc['original', 'mean']
regression_supported = regression_summary.loc['Forest Sketch', 'r2'] >= regression_summary.loc['original', 'r2']
print(f'Classification hypothesis: {"SUPPORTED" if classification_supported else "NOT SUPPORTED"}')
print(f'Regression hypothesis: {"SUPPORTED" if regression_supported else "NOT SUPPORTED"}')

,seed,method,r2,rmse
0,0,original,0.990,18.380
1,0,initial random projection,0.632,112.762
2,0,one-shot tree-path,0.372,147.202
3,0,Forest Sketch,0.116,174.709
4,1,original,0.995,18.037
5,1,initial random projection,0.297,211.211
6,1,one-shot tree-path,0.211,223.698
7,1,Forest Sketch,0.119,236.411


,r2,rmse
method,,
Forest Sketch,0.117,205.560
initial random projection,0.464,161.986
one-shot tree-path,0.292,185.450
original,0.993,18.208


Classification hypothesis: NOT SUPPORTED
Regression hypothesis: NOT SUPPORTED
